# Dev notebook to generate figures for animation of HH domain

In [1]:
## import required packages
import numpy as np
import pandas as pd
import sys
import xarray as xr
import matplotlib.pyplot as plt
import glob
import warnings
from pathlib import Path
import s3fs
from pprint import pprint
import matplotlib
from matplotlib.backends.backend_pdf import PdfPages
from tqdm import tqdm
import cmocean
import os
import subprocess

In [5]:
from dask.distributed import Client

client = Client("tcp://127.0.0.1:43937")
client

<Client: 'tcp://127.0.0.1:43937' processes=8 threads=32, memory=123.95 GiB>

### Open 3D model fields

In [6]:
# function to open zarr store with a provided s3 bucket path

def open_zarr_store(s3_path):
    # initalize s3 file system
    s3 = s3fs.S3FileSystem(anon=False)

    # define location of zarr store and open
    store = s3fs.S3Map(root=s3_path, s3=s3, check=False)
    zarr_store = xr.open_zarr(store)
    
    return zarr_store

In [7]:
# open SALT zarr store
salt_zarr = open_zarr_store('s3://ecco-processed-data/SASSIE/N1/HH/ZARR/SALT_AVG_DAILY.ZARR/')

# open SEA ICE AREA zarr store
SIarea_zarr = open_zarr_store('s3://ecco-processed-data/SASSIE/N1/HH/ZARR/SIarea_AVG_DAILY.ZARR/')

***

## Generate series of PNG images, one for each day

### SALINITY

In [20]:
time_start = "2020-01-01"
time_end = "2020-12-31"
k_level = 0

In [21]:
salt_da = salt_zarr.sel(time=slice(time_start,time_end)).isel(k=k_level).SALT

In [22]:
for i in tqdm(range(len(salt_da.time))):
    
    # pull out one day
    salt_day_da = salt_da.isel(time=i)
    
    # make plot
    plt.rcParams.update({'font.size': 9})
    fig, ax = plt.subplots(1,1,figsize=[12,8])
    pc = salt_day_da.plot(ax=ax,vmin=20,vmax=36,cmap='turbo', add_colorbar=False)
    ax.set_title("");
    
    # make background black
    ax.set_facecolor('k')
    
    # remove tick labels
    ax.set_xticklabels([])
    ax.set_yticklabels([]);
    ax.tick_params(left = False, bottom = False) 
    ax.set_xlabel("")
    ax.set_ylabel("");

    # --- Create colorbar axes (small horizontal bar in bottom left) ---
    # [left, bottom, width, height] in figure fraction (0 to 1)
    cbar_ax = fig.add_axes([0.23, 0.17, 0.13, 0.015])  # Adjust position and size here
    
    # Create the colorbar
    cbar = fig.colorbar(pc, cax=cbar_ax, orientation='horizontal', extend='both')
    cbar.set_ticks([20, 24, 28, 32, 36])
    cbar.set_label('Salinity (g/kg)', fontsize=9, color='white')

    # add text for date
    date_str = str(salt_day_da.time.values)
    date = date_str[5:10] + '-' + date_str[0:4]
    # ax.text(320,950,f'{date}',color='white')
    
    # Date text above colorbar
    fig.text(0.248, 0.21, f'{date}', color='white', fontsize=12, ha='left')
    
    # Optional: Make colorbar outline white for visibility on dark background
    cbar.outline.set_edgecolor('white')
    cbar.ax.xaxis.set_tick_params(color='white')
    plt.setp(cbar.ax.xaxis.get_ticklabels(), color='white');

    # save fig
    filename = f"sassie-ecco-salinity_{i:04d}.png"
    plt.savefig(f"/home/jpluser/efs-mount-point/mzahn/sassie/HH/animation/pngs/{filename}",dpi=300,bbox_inches='tight')
    
    plt.close()
    
    # print(f'. . . saved {filename}')

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 366/366 [20:17<00:00,  3.33s/it]


### SEA ICE

In [23]:
time_start = "2020-01-01"
time_end = "2020-12-31"

In [24]:
seaice_da = SIarea_zarr.sel(time=slice(time_start,time_end)).SIarea

In [25]:
for i in tqdm(range(len(seaice_da.time))):
    
    # pull out one day
    seaice_day_da = seaice_da.isel(time=i)
    
    # make plot
    plt.rcParams.update({'font.size': 9})
    fig, ax = plt.subplots(1,1,figsize=[12,8])
    pc = (seaice_day_da*100).plot(ax=ax,vmin=0,vmax=100,cmap=cmocean.cm.ice,add_colorbar=False)
    ax.set_title("");
    
    # make background black
    ax.set_facecolor('lightgray')
    
    # remove tick labels
    ax.set_xticklabels([])
    ax.set_yticklabels([]);
    ax.tick_params(left = False, bottom = False) 
    ax.set_xlabel("")
    ax.set_ylabel("");

    # --- Create colorbar axes (small horizontal bar in bottom left) ---
    # [left, bottom, width, height] in figure fraction (0 to 1)
    cbar_ax = fig.add_axes([0.23, 0.17, 0.13, 0.015])  # Adjust position and size here
    
    # Create the colorbar
    cbar = fig.colorbar(pc, cax=cbar_ax, orientation='horizontal')
    # cbar.set_ticks([20, 24, 28, 32, 36])
    cbar.set_label('Sea Ice Concentration (%)', fontsize=9, color='k')

    # add text for date
    date_str = str(seaice_day_da.time.values)
    date = date_str[5:10] + '-' + date_str[0:4]
    
    # Date text above colorbar
    fig.text(0.248, 0.21, f'{date}', color='k', fontsize=12, ha='left')
    
    # Optional: Make colorbar outline white for visibility on dark background
    cbar.outline.set_edgecolor('k')
    cbar.ax.xaxis.set_tick_params(color='k')
    plt.setp(cbar.ax.xaxis.get_ticklabels(), color='k');

    # save fig
    filename = f"sassie-ecco-seaice-2020_{i:04d}.png"
    plt.savefig(f"/home/jpluser/efs-mount-point/mzahn/sassie/HH/animation/pngs/{filename}",dpi=300,bbox_inches='tight')
    
    plt.close()
    
    # print(f'. . . saved {filename}')

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 366/366 [16:36<00:00,  2.72s/it]


### SALINITY AND SEA ICE

In [ ]:
# Define directories
# png_dir = "/home/jpluser/efs-mount-point/mzahn/sassie/HH/animation/pngs/"
png_dir = '/nvme_data2/pngs/'

# Loop over years
# for year in range(2014, 2021):
for year in range(2015, 2020):
    time_start = f"{year}-01-01"
    time_end = f"{year}-12-31"
    k_level = 0

    print(f"Processing year {year} ...")

    seaice_da = SIarea_zarr.sel(time=slice(time_start,time_end)).SIarea*100
    salt_da = salt_zarr.sel(time=slice(time_start,time_end)).isel(k=k_level).SALT

    SIarea_over20 = seaice_da.where(seaice_da > 20, np.nan)
    salt_ice_mask = salt_da.where(seaice_da < 20, np.nan)

    # Generate PNG frames
    for i in tqdm(range(len(seaice_da.time))):
        salt_day_da = salt_ice_mask.isel(time=i)
        seaice_day_da = SIarea_over20.isel(time=i)

        plt.rcParams.update({'font.size': 9})
        fig, ax = plt.subplots(1,1,figsize=[12,8])
        salt_day_da.plot(ax=ax, vmin=20, vmax=36, cmap='turbo', add_colorbar=False)
        seaice_day_da.plot(ax=ax, vmin=0, vmax=100, cmap=cmocean.cm.ice, add_colorbar=False)
        ax.set_title("")
        ax.set_facecolor('k')
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        ax.tick_params(left=False, bottom=False)
        ax.set_xlabel("")
        ax.set_ylabel("")

        date_str = str(salt_day_da.time.values)
        date = date_str[5:10] + '-' + date_str[0:4]
        fig.text(0.248, 0.15, f'{date}', color='white', fontsize=12, ha='left')

        filename = f"sassie-ecco-seaice-salinity-{year}_{i:04d}.png"
        filepath = os.path.join(png_dir, filename)
        plt.savefig(filepath, dpi=300, bbox_inches='tight')
        plt.close()

    print("Running ffmpeg to create mp4 ...")
    ffmpeg_cmd = [
        "ffmpeg",
        "-r", "10",
        "-i", os.path.join(png_dir, f"sassie-ecco-seaice-salinity-{year}_%04d.png"),
        "-pix_fmt", "yuv420p",
        "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2",
        "-crf", "10",
        os.path.join(png_dir, f"sassie-ecco-model-seaice-salinity-{year}.mp4")
    ]
    subprocess.run(ffmpeg_cmd, check=True)

    print("Uploading mp4 to S3 ...")
    s3_cmd = [
        "aws", "s3", "cp",
        os.path.join(png_dir, f"sassie-ecco-model-seaice-salinity-{year}.mp4"),
        f"s3://ecco-processed-data/SASSIE/Videos/"
    ]
    subprocess.run(s3_cmd, check=True)

    print("Cleaning up PNG files ...")
    # Remove PNGs for this year
    for i in range(len(seaice_da.time)):
        filepath = os.path.join(png_dir, f"sassie-ecco-seaice-salinity-{year}_{i:04d}.png")
        if os.path.exists(filepath):
            os.remove(filepath)

    print(f"Year {year} done.\n")

Processing year 2015 ...


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 365/365 [30:03<00:00,  4.94s/it]
ffmpeg version 6.1.1 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 12.3.0 (conda-forge gcc 12.3.0-5)
  configuration: --prefix=/home/conda/feedstock_root/build_artifacts/ffmpeg_1710227007179/_h_env_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_plac --cc=/home/conda/feedstock_root/build_artifacts/ffmpeg_1710227007179/_build_env/bin/x86_64-conda-linux-gnu-cc --cxx=/home/conda/feedstock_root/build_artifacts/ffmpeg_1710227007179/_build_env/bin/x86_64-conda-linux-gnu-c++ --nm=/home/conda/feedstock_root/build_artifacts/ffmpeg_1710227007179/_build_env/bin/x86_64-conda-linux-gnu-nm --ar=/home/conda/feedstock_root/build_artifacts/ffmpeg_1710227007179/_build_env/bin/x86_64-co

Running ffmpeg to create mp4 ...


[libx264 @ 0x56359fdc49c0] using SAR=1906/1907
[libx264 @ 0x56359fdc49c0] using cpu capabilities: MMX2 SSE2Fast SSSE3 SSE4.2 AVX FMA3 BMI2 AVX2 AVX512
[libx264 @ 0x56359fdc49c0] profile High, level 5.0, 4:2:0, 8-bit
[libx264 @ 0x56359fdc49c0] 264 - core 164 r3095 baee400 - H.264/MPEG-4 AVC codec - Copyleft 2003-2022 - http://www.videolan.org/x264.html - options: cabac=1 ref=3 deblock=1:0:0 analyse=0x3:0x113 me=hex subme=7 psy=1 psy_rd=1.00:0.00 mixed_ref=1 me_range=16 chroma_me=1 trellis=1 8x8dct=1 cqm=0 deadzone=21,11 fast_pskip=1 chroma_qp_offset=-2 threads=48 lookahead_threads=8 sliced_threads=0 nr=0 decimate=1 interlaced=0 bluray_compat=0 constrained_intra=0 bframes=3 b_pyramid=2 b_adapt=1 b_bias=0 direct=1 weightb=1 open_gop=0 weightp=2 keyint=250 keyint_min=10 scenecut=40 intra_refresh=0 rc_lookahead=40 rc=crf mbtree=1 crf=10.0 qcomp=0.60 qpmin=0 qpmax=69 qpstep=4 ip_ratio=1.40 aq=1:1.00
Output #0, mp4, to '/nvme_data2/pngs/sassie-ecco-model-seaice-salinity-2015.mp4':
  Metadata:

Uploading mp4 to S3 ...
upload: ../../../../../../nvme_data2/pngs/sassie-ecco-model-seaice-salinity-2015.mp4 to s3://ecco-processed-data/SASSIE/Videos/sassie-ecco-model-seaice-salinity-2015.mp4
Cleaning up PNG files ...
Year 2015 done.

Processing year 2016 ...


 47%|███████████████████████████████████████████████████████████████                                                                        | 171/366 [14:08<16:49,  5.18s/it]